### Test Envirnment 

- Step 1 → API connection
- Step 2 → PDF text extraction
- Step 3 → Text chunking
- Step 4 → Embeddings API
- Step 5 → FAISS vector index
- Step 6 → Retrieval
- Step 7 → RAG prompt
- Step 8 → LLM answer

#### Step 1 → API connection

In [1]:
from dotenv import load_dotenv
import os
from huggingface_hub import InferenceClient

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError("HF_TOKEN not found in .env")


client = InferenceClient(api_key=HF_TOKEN)


In [2]:
messages = [
    {   "role" : "user",
        "content" : "explain what is RAG in one line ?"
    }
]

response = client.chat_completion(
    model="MiniMaxAI/MiniMax-M2.5",
    messages=messages,
    max_tokens=50
)

print(response)

ChatCompletionOutput(choices=[ChatCompletionOutputComplete(finish_reason='length', index=0, message=ChatCompletionOutputMessage(role='assistant', content='**RAG (Retrieval-Augmented Generation)** is an AI technique that combines a language model with a external knowledge base to retrieve', reasoning=None, tool_call_id=None, tool_calls=None, reasoning_content='The user is asking for a brief explanation of RAG (Retrieval-Augmented Generation) in one line.'), logprobs=None, token_ids=None)], created=1773312666, id='6a656035-e963-4cb5-a7d0-418c73f57f6d', model='accounts/fireworks/models/minimax-m2p5', system_fingerprint=None, usage=ChatCompletionOutputUsage(completion_tokens=50, prompt_tokens=46, total_tokens=96, prompt_tokens_details={'cached_tokens': 0}), object='chat.completion', prompt_token_ids=None)


In [3]:
import json 
print(json.dumps(response, indent=4))


{
    "choices": [
        {
            "finish_reason": "length",
            "index": 0,
            "message": {
                "role": "assistant",
                "content": "**RAG (Retrieval-Augmented Generation)** is an AI technique that combines a language model with a external knowledge base to retrieve",
                "reasoning": null,
                "tool_call_id": null,
                "tool_calls": null
            },
            "logprobs": null
        }
    ],
    "created": 1773312666,
    "id": "6a656035-e963-4cb5-a7d0-418c73f57f6d",
    "model": "accounts/fireworks/models/minimax-m2p5",
    "usage": {
        "completion_tokens": 50,
        "prompt_tokens": 46,
        "total_tokens": 96
    },
    "system_fingerprint": null,
    "object": "chat.completion",
    "prompt_token_ids": null
}


In [4]:
print(response.choices[0].message.content)

**RAG (Retrieval-Augmented Generation)** is an AI technique that combines a language model with a external knowledge base to retrieve


#### Step 2 → PDF text extraction

In [5]:
import fitz #PyMuPdf
import os 

pdf_path = "D:/Code/Python/Intership_Learning/AI_Ml_Projects/RAG_Q&A_Bot/PDFs/Introduction to Outlier.docx.pdf"

doc = fitz.open(pdf_path) #open in PyMuPdf

pdf_text = ""

for page in doc:
    pdf_text+= page.get_text() + "\n"

page_no = doc.page_count
doc.close()

print(f"PDF Loaded Successfully. \nDocument Total Pages : {page_no}\nNumber of character extracted : {len(pdf_text)}\n")


PDF Loaded Successfully. 
Document Total Pages : 22
Number of character extracted : 18680



#### Note :
- Sometimes PyMuPDF DLL causes import errors on Windows 11. 

- To fix: go to Defender → Virus & Threat Protection → Ransomware Protection → Manage Ransomware Protection → Allow an app through Controlled Folder Access → add your python.exe file.

- Restart VS code editor or Kernal

#### Step 3 → Text chunking

In [6]:
# Logic of chunk :
def chunk_text(text, size = 500, overlap=50):
    """
    Splits text into overlapping chunks.
    text: full string
    size: characters per chunk
    overlap: characters to overlap between chunks
    """

    chunks = []
    start = 0

    while start < len(text):
        end = start + size
        chunk = text[start:end]
        chunks.append(chunk)

        start+= size - overlap #for continuity

    return chunks

# Create Chunks:
chunks = chunk_text(pdf_text, size = 500, overlap = 50)

print(f"PDF split into {len(chunks)} chunks.")
print(f"Example chunk :\n{chunks[0]}")


PDF split into 42 chunks.
Example chunk :
Introduction to Outlier 
An outlier is an observation that is unlike the other 
observations. It is rare, or distinct, or does not fit in some way. 
It is also called anomalies. 
 
Outliers are extreme values or unusual data points in a dataset 
that differ significantly from other observations. They are 
crucial to understand because they can affect model accuracy 
and lead to misleading insights if not properly addressed.  
Types of Outliers 
●​ Univariate Outliers: These are unusual values in


#### Step 4 → Embeddings API

In [7]:
# from sentence_transformers import SentenceTransformer
# import numpy as np

In [8]:
# embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# def get_vac(text):
#     """
#     Get embedding vector for a text string using HF API.
#     Returns a numpy array.
#     """
#     vactore = embedding_model.encode(text)
#     return np.array(response["embeddings"], dtype=np.float32)

# embeddings = []

# for chunk in chunks:
#     vac = get_vac(chunk)
#     embeddings.append(vac) #return (384,)

# embeddings = np.vstack(embeddings)
# print(f"Embeddings generated for all chunks. \nShape. {embeddings.shape}")


In [9]:
# import numpy as np
# from dotenv import load_dotenv
# import os
# from huggingface_hub import InferenceClient

# load_dotenv()
# HF_TOKEN = os.getenv("HF_TOKEN")

# if HF_TOKEN is None:
#     raise ValueError("HF_TOKEN not found in .env")


# client = InferenceClient(api_key=HF_TOKEN)


In [10]:
# # Logic of embeddings generate:
# embedding_model = "sentence-transformers/all-MiniLM-L6-v2"

# def get_embeddings(text):
#     """
#     Get embedding vector for a text string using HF API.
#     Returns a numpy array.
#     """
#     response = client.text_embeddings(model=embedding_model, input = text)
#     vectore = np.array(response["embeddings"], dtype=np.float32)
#     return response

# embeddings = []
# for chunk in chunks:
#     vec = get_embeddings(chunk)
#     embeddings.append(vec)

# embeddings = np.vstack(embeddings)
# print(f"Embeddings generated for all chunks. \nShape. {embeddings.shape}")

#### Step 5 → FAISS vector index